# Tahap 06: Model Evaluation & Testing

Tahap terakhir ini bertujuan untuk menguji performa model *Multinomial Logistic Regression* menggunakan data pengujian (*testing data*). Evaluasi dilakukan dengan membandingkan hasil prediksi model terhadap label asli untuk menghasilkan metrik performa seperti *Accuracy*, *Classification Report* (Presisi, Recall, F1-Score), serta visualisasi *Confusion Matrix*.

In [ ]:
# --- LIBRARIES ---
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. KONFIGURASI DIREKTORI & PATH ---
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'

print("="*60)
print("TAHAP EVALUASI: TESTING MULTINOMIAL LOGISTIC REGRESSION")
print("="*60)

# --- 2. PEMUATAN MODEL, MAPPING, & DATA TESTING ---
# Memuat Model Logistic Regression dan Mapping yang telah dilatih
with open(os.path.join(BASE_DIR, 'model_multinomial_lr_2.0.pkl'), 'rb') as f:
    model = pickle.load(f)

with open(os.path.join(BASE_DIR, 'custom_mapping_2.0.pkl'), 'rb') as f:
    custom_mapping = pickle.load(f)

# Memuat Data Testing (Matriks Fitur dan Data Tabular)
X_test_selected = sparse.load_npz(os.path.join(BASE_DIR, 'X_test_selected_2.0.npz'))
test_df = pd.read_csv(os.path.join(BASE_DIR, 'data_test_final_2.0.csv'))

# --- 3. PREPARASI LABEL TESTING (Aktual) ---
# Menerapkan mapping: {'keluhan': 0, 'saran': 1, 'pujian': 2}
y_true = test_df['label_pks'].map(custom_mapping)

## 6.1 Prediksi dan Kalkulasi Metrik Evaluasi

In [ ]:
# --- 4. PROSES PREDIKSI ---
print("Proses pengujian model pada data testing sedang berlangsung...")
y_pred = model.predict(X_test_selected)
print("Proses prediksi selesai dilakukan.")
print("-" * 60)

# --- 5. EVALUASI METRIK KINERJA ---
acc = accuracy_score(y_true, y_pred)
print(f"AKURASI KESELURUHAN: {acc:.2%}")
print("\nCLASSIFICATION REPORT:")

# Definisi nama target sesuai dengan urutan index (0, 1, 2)
target_names = ['Keluhan (0)', 'Saran (1)', 'Pujian (2)']
print(classification_report(y_true, y_pred, target_names=target_names))
print("=" * 60)

## 6.2 Visualisasi Confusion Matrix

In [ ]:
# --- 6. VISUALISASI CONFUSION MATRIX ---
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)

plt.xlabel('Prediksi Model (Predicted Label)')
plt.ylabel('Jawaban Aktual (True Label)')
plt.title('Confusion Matrix - Logistic Regression')

# Menampilkan grafik
plt.show()

## 6.3 Penyimpanan Laporan Hasil Prediksi

In [ ]:
# --- 7. PENYIMPANAN HASIL PREDIKSI KE CSV ---
# Mengonversi nilai numerik kembali ke teks label untuk kemudahan analisis manual
reverse_mapping = {v: k for k, v in custom_mapping.items()}

test_df['prediksi_angka'] = y_pred
test_df['prediksi_label'] = test_df['prediksi_angka'].map(reverse_mapping)

# Menandai status kebenaran prediksi setiap baris
test_df['status'] = test_df.apply(
    lambda x: 'BENAR' if x['label_pks'] == x['prediksi_label'] else 'SALAH', axis=1
)

# Menyimpan ke dalam file CSV
output_file = os.path.join(BASE_DIR, 'hasil_prediksi_testing_2.0.csv')
test_df.to_csv(output_file, index=False)

print(f"Data hasil pengujian secara detail per baris telah disimpan di: {output_file}")
print("="*60)